# Chapter 6: Evaluating GenAI Applications within MLflow

This chapter focuses on practical techniques for evaluating Generative AI (GenAI) applications using MLflow. You will learn how to assess GenAI models—such as customer service agents—using modern evaluation strategies, metrics, and workflows tailored for natural language generation tasks.

## Setup

Before running the evaluation workflow, ensure your environment is properly configured:
* Install all required Python packages listed in the requirements file
* Restart the Python kernel to activate newly installed packages and avoid dependency conflicts

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
# Set the Unity Catalog and schema for Unity Airways data and models
# These variables are used to reference tables and models throughout the notebook
CATALOG = 'workspace'
SCHEMA = 'unity_airways'

### Why Traditional Metrics Miss GenAI Application Behaviour

GenAI applications represent a paradigm shift from traditional ML models. Unlike classic models that produce structured, deterministic outputs, GenAI systems generate natural language responses that must be evaluated for qualities like relevance, factual accuracy, and groundedness.

Traditional ML evaluation metrics (accuracy, precision, recall, F1-score) were designed for classification and regression tasks with clear ground truth labels. These metrics fall short for GenAI applications because:

1. **Deterministic vs. Generative Outputs**: Traditional models produce fixed outputs for given inputs, while GenAI models generate variable, creative responses
2. **Structured vs. Unstructured Data**: Traditional metrics work with numerical or categorical outputs, not natural language text
3. **Single Correct Answer vs. Multiple Valid Responses**: GenAI tasks often have many acceptable answers, making binary accuracy insufficient
4. **Context and Nuance**: Traditional metrics don't capture semantic meaning, tone, helpfulness, or user experience
5. **Safety and Ethics**: GenAI outputs must be evaluated for harmful content, bias, and policy compliance

### What are the Components to Evaluate

GenAI applications typically consist of multiple components that require different evaluation approaches:

1. **Retrieval Components**: 
   - Retrieval accuracy and relevance
   - Document ranking quality
   - Coverage of relevant information

2. **Generation Components**:
   - Factual correctness and groundedness
   - Relevance to user query
   - Coherence and fluency
   - Tone and style appropriateness

3. **End-to-End System**:
   - User experience and satisfaction
   - Task completion effectiveness
   - Safety and policy compliance
   - Latency and performance

4. **Business Logic**:
   - Adherence to guidelines and policies
   - Consistency across similar queries
   - Integration with downstream systems

### Evaluation Modes: Direct Evaluation vs Answer Sheet Evaluation

**Direct Evaluation:**
- Assesses model outputs directly against criteria or guidelines
- Uses LLM judges to evaluate qualities like helpfulness, relevance, safety
- Suitable for open-ended tasks without single correct answers
- Examples: Chatbot responses, creative writing, summarization

**Answer Sheet Evaluation:**
- Compares model outputs to curated reference answers
- Uses exact match, semantic similarity, or custom comparison functions
- Suitable for tasks with clear correct answers
- Examples: Question answering, fact extraction, classification

## Use Case: Unity Airways Customer Service Agent

This notebook demonstrates best practices for evaluating a GenAI-powered customer service agent for Unity Airways using MLflow 3.x. You will learn how to:
* Set up and load a Retrieval-Augmented Generation (RAG) chain model
* Prepare a realistic evaluation dataset covering diverse customer service scenarios
* Define and implement a comprehensive suite of evaluation scorers, including LLM-based, guidelines-based, and code-based metrics
* Run and interpret GenAI evaluation workflows in MLflow, focusing on qualities such as relevance, correctness, safety, brand compliance, and completeness

The workflow follows the concepts discussed in Chapter 4, providing hands-on code and explanations for each step. This notebook is designed to be a practical reference for evaluating GenAI applications in real-world customer service contexts.

### Step 1: Load the RAG Chatbot

In [0]:
MODEL_VERSION = "1"

In [0]:
import mlflow

model_uri = f"models:/{CATALOG}.{SCHEMA}.rag_chain_model/{MODEL_VERSION}"
loaded_chain = mlflow.langchain.load_model(model_uri)

In [0]:
from mlflow.models import ModelConfig

rag_chain_conf_path = "../conf/chapter04_conf.yml"
model_config = ModelConfig(development_config=rag_chain_conf_path)

In [0]:
model_config.get("input_example")

In [0]:
# Test the chain locally

loaded_chain.invoke(model_config.get("input_example"))

### Step 2: Import Evaluation Dataset

Now let's import an evaluation dataset that covers different types of customer service scenarios.

In [0]:
# Read evaluation data from Unity Airways QA dataset in Unity Catalog
eval_data_df = spark.read.table(
    f"{CATALOG}.{SCHEMA}.qa_dataset"
)

# Transform columns to match agent evaluation schema
eval_data = eval_data_df.selectExpr(
    "struct(Question as customer_question) as inputs",
    "struct(Answer as expected_response) as expectations"
)

# Sample 20 examples for evaluation and convert to Pandas DataFrame
eval_df = eval_data.limit(20).toPandas()

# Display the sampled evaluation dataset
display(eval_df)

### Step 3: Define Comprehensive Scorers

Let's create a mix of LLM-based and code-based scorers to evaluate our customer service agent comprehensively.


In [0]:
# Define comprehensive scorers for Unity Airways customer service evaluation

from mlflow.genai.scorers import RelevanceToQuery, Correctness, Safety, RetrievalGroundedness, RetrievalRelevance, RetrievalSufficiency, Guidelines, scorer
from mlflow.entities import Feedback

# Custom Guidelines-Based Scorers
professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="""
    The customer service response must use professional, courteous language appropriate for airline customer service.
    Requirements:
    - Use polite and respectful language
    - Avoid casual expressions or slang
    - Maintain helpful and solution-oriented tone
    - Include appropriate greetings/closings when relevant
    """
)

brand_compliance_scorer = Guidelines(
    name="brand_compliance",
    guidelines="""
    The response must follow Unity Airways brand guidelines:
    - Mention Unity Airways when appropriate
    - Use consistent contact information (1-800-UNITY-AIR, unityairways.com)
    - Maintain positive, customer-focused messaging
    - Provide specific, actionable information when possible
    """
)

completeness_scorer = Guidelines(
    name="response_completeness", 
    guidelines="""
    The customer service response must completely address the customer's question:
    - Directly answer the specific question asked
    - Provide all relevant details mentioned in the expected response
    - Include next steps or additional resources when appropriate
    - Avoid generic responses when specific information is requested
    """
)

# Code-Based Scorers
@scorer
def response_length_checker(outputs) -> Feedback:
    """Check if response length is appropriate (not too short or too long)."""
    response = outputs if isinstance(outputs, str) else outputs.get('response', '')
    word_count = len(response.split())
    
    if word_count < 5:
        return Feedback(value = 0, rationale = f"Response too short ({word_count} words)")
    elif word_count > 100:
        return Feedback(value =  0.5, rationale = f"Response quite long ({word_count} words)")
    else:
        return Feedback(value =  1, rationale = f"Appropriate length ({word_count} words)")

@scorer
def contact_info_checker(outputs) -> Feedback:
    """Check if response includes appropriate Unity Airways contact information."""
    response = outputs if isinstance(outputs, str) else outputs.get('response', '')
    response_lower = response.lower()
    
    has_phone = "1-800-unity-air" in response_lower
    has_website = "unityairways.com" in response_lower
    mentions_contact = any(word in response_lower for word in ["call", "phone", "website", "visit"])
    
    if has_phone and has_website:
        return Feedback(value = 1, rationale = "Includes both phone and website")
    elif has_phone or has_website:
        return Feedback(value = 0.8, rationale = "Includes one form of contact info")
    elif mentions_contact:
        return Feedback(value = 0.3, rationale = "Mentions contacting but no specific info")
    else:
        return Feedback(value = 0, rationale = "No contact information provided")

@scorer
def policy_accuracy_checker(outputs, expectations=None) -> Feedback:
    """Check if the response contains accurate policy information."""
    if not expectations:
        return Feedback(value = 0.5, rationale = "No expected response to compare against")
    
    response = outputs if isinstance(outputs, str) else outputs.get('response', '')
    expected_response = expectations.get('expected_response', '') if isinstance(expectations, dict) else str(expectations)
    
    # Simple keyword overlap check
    response_words = set(response.lower().split())
    expected_words = set(expected_response.lower().split())
    
    # Remove common words
    common_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are'}
    response_words -= common_words
    expected_words -= common_words
    
    if not expected_words:
        return Feedback(value = 0.5, rationale = "No meaningful expected words to compare")
    
    overlap = len(response_words & expected_words) / len(expected_words)
    
    if overlap >= 0.7:
        return Feedback(value = 1, rationale = f"High accuracy ({overlap:.2f} keyword overlap)")
    elif overlap >= 0.4:
        return Feedback(value = 0.7, rationale = f"Good accuracy ({overlap:.2f} keyword overlap)")
    elif overlap >= 0.2:
        return Feedback(value = 0.4, rationale = f"Partial accuracy ({overlap:.2f} keyword overlap)")
    else:
        return Feedback(value = 0, rationale = f"Low accuracy ({overlap:.2f} keyword overlap)")

# Combine all scorers
unity_airways_scorers = [
    # LLM-based scorers
    RelevanceToQuery(), 
    Correctness(), 
    Safety(), 
    RetrievalGroundedness(), 
    RetrievalRelevance(), 
    RetrievalSufficiency(),

    # Custom guidelines-based scorers
    professional_tone_scorer,
    brand_compliance_scorer,
    completeness_scorer,
    
    # Code-based scorers  
    response_length_checker,
    contact_info_checker,
    policy_accuracy_checker
]

print(f"Created {len(unity_airways_scorers)} scorers for Unity Airways evaluation:")
for scorer in unity_airways_scorers:
    print(f"- {scorer.name}: {'LLM-based' if hasattr(scorer, 'guidelines') or scorer.name in ['relevance_to_query', 'safety'] else 'Code-based'}")

# Test a scorer
test_response = "One carry-on bag (22x14x9 inches) and one personal item allowed. Is there anything else I can help you with today?"
test_expected = {"expected_response": "One carry-on bag (22x14x9 inches) and one personal item allowed"}

accuracy_result = policy_accuracy_checker(test_response, test_expected)
print(f"\nTest scorer result: {accuracy_result}")


### Step 4: Run Complete Evaluation and Analysis

Now let's run a complete evaluation of our Unity Airways customer service agent and analyze the results comprehensively.


In [0]:
def predict_fn(customer_question):
    return loaded_chain.invoke({
        "messages": [
            {"role": "user", "content": customer_question}
        ]
    })

eval_results = mlflow.genai.evaluate(
    data=eval_df,
    predict_fn=predict_fn,
    scorers=unity_airways_scorers
)